## Phase 1: Original data load (documentation)

This is the code I ran interactively to clean and upload the CSVs to SQL Server.
It was recovered from my Jupyter history. It is shown here as documentation only
and is not meant to be re-run, because it appends to the tables.

Note: the `transactions` table here shows 250,000 rows because the upload was run twice.
I detected this by comparing the row count to the source file (125,000) and fixed it
in Phase 2 by truncating and reloading.

```python
import pandas as pd
from sqlalchemy import create_engine, text

# CONNECTION
SERVER   = "localhost\\SQLEXPRESS"
DATABASE = "retail_sales"

engine = create_engine(
    f"mssql+pyodbc://{SERVER}/{DATABASE}"
    f"?driver=ODBC+Driver+17+for+SQL+Server"
    f"&trusted_connection=yes"
)

# LOAD CSVs
transactions = pd.read_csv(r"C:\Users\Asus\Downloads\archive\Retail_Data_Transactions.csv")
response     = pd.read_csv(r"C:\Users\Asus\Downloads\archive\Retail_Data_Response.csv")

# CLEAN TRANSACTIONS
transactions["trans_date"]  = pd.to_datetime(transactions["trans_date"], format="%d-%b-%y")
transactions["tran_amount"] = pd.to_numeric(transactions["tran_amount"], errors="coerce")
transactions["customer_id"] = transactions["customer_id"].astype(str).str.strip()
transactions.dropna(inplace=True)
transactions = transactions[transactions["tran_amount"] > 0]
print(f"Transactions: {len(transactions):,} rows")

# CLEAN RESPONSE
response["customer_id"] = response["customer_id"].astype(str).str.strip()
response["response"]    = pd.to_numeric(response["response"], errors="coerce").fillna(0).astype(int)
print(f"Response: {len(response):,} rows")

# UPLOAD
transactions.to_sql("transactions",  engine, if_exists="append", index=False, chunksize=1000)
response.to_sql("customer_response", engine, if_exists="append", index=False, chunksize=1000)

# VERIFY
with engine.connect() as conn:
    t = conn.execute(text("SELECT COUNT(*) FROM transactions")).scalar()
    r = conn.execute(text("SELECT COUNT(*) FROM customer_response")).scalar()
    print(f"transactions      : {t:,} rows in DB")
    print(f"customer_response : {r:,} rows in DB")
```

## Table definitions (DDL)

Created in SSMS before loading the data.

```sql

-- transactions
USE [retail_sales]
GO

/****** Object:  Table [dbo].[transactions]    Script Date: 10-10-2026 7.09.15 PM ******/
SET ANSI_NULLS ON
GO

SET QUOTED_IDENTIFIER ON
GO

CREATE TABLE [dbo].[transactions](
	[transaction_id] [int] IDENTITY(1,1) NOT NULL,
	[customer_id] [varchar](20) NOT NULL,
	[trans_date] [date] NOT NULL,
	[tran_amount] [decimal](10, 2) NOT NULL,
	[trans_month]  AS (datepart(month,[trans_date])) PERSISTED,
	[trans_year]  AS (datepart(year,[trans_date])) PERSISTED,
PRIMARY KEY CLUSTERED 
(
	[transaction_id] ASC
)WITH (PAD_INDEX = OFF, STATISTICS_NORECOMPUTE = OFF, IGNORE_DUP_KEY = OFF, ALLOW_ROW_LOCKS = ON, ALLOW_PAGE_LOCKS = ON, OPTIMIZE_FOR_SEQUENTIAL_KEY = OFF) ON [PRIMARY]
) ON [PRIMARY]
GO



-- customer_response
USE [retail_sales]
GO

/****** Object:  Table [dbo].[customer_response]    Script Date: 10-10-2026 7.11.59 PM ******/
SET ANSI_NULLS ON
GO

SET QUOTED_IDENTIFIER ON
GO

CREATE TABLE [dbo].[customer_response](
	[customer_id] [varchar](20) NOT NULL,
	[response] [tinyint] NOT NULL,
PRIMARY KEY CLUSTERED 
(
	[customer_id] ASC
)WITH (PAD_INDEX = OFF, STATISTICS_NORECOMPUTE = OFF, IGNORE_DUP_KEY = OFF, ALLOW_ROW_LOCKS = ON, ALLOW_PAGE_LOCKS = ON, OPTIMIZE_FOR_SEQUENTIAL_KEY = OFF) ON [PRIMARY]
) ON [PRIMARY]
GO


```

In [2]:
import subprocess
subprocess.run(["pip", "install", "sqlalchemy", "pyodbc"], check=True)
print("Done!")

Done!


In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

# CONNECTION
SERVER   = "localhost\\SQLEXPRESS"
DATABASE = "retail_sales"
engine   = create_engine(
    f"mssql+pyodbc://{SERVER}/{DATABASE}"
    f"?driver=ODBC+Driver+17+for+SQL+Server"
    f"&trusted_connection=yes"
)

# LOAD FROM DB
print("Loading data from SQL Server...")
transactions = pd.read_sql("SELECT * FROM transactions", engine)
response     = pd.read_sql("SELECT * FROM customer_response", engine)

print(f"transactions : {len(transactions):,} rows")
print(f"response     : {len(response):,} rows")

# MISSING VALUES
print("\nMissing values in transactions:")
print(transactions.isnull().sum())

print("\nMissing values in customer_response:")
print(response.isnull().sum())

# DUPLICATES
dup_t = transactions.duplicated().sum()
dup_r = response.duplicated().sum()
print(f"\nDuplicate rows in transactions     : {dup_t}")
print(f"Duplicate rows in customer_response: {dup_r}")

Loading data from SQL Server...
transactions : 250,000 rows
response     : 6,884 rows

Missing values in transactions:
transaction_id    0
customer_id       0
trans_date        0
tran_amount       0
trans_month       0
trans_year        0
dtype: int64

Missing values in customer_response:
customer_id    0
response       0
dtype: int64

Duplicate rows in transactions     : 0
Duplicate rows in customer_response: 0


In [7]:
import os
# Find the CSV files automatically
for root, dirs, files in os.walk(r"C:\Users"):
    for file in files:
        if "Retail_Data" in file:
            print(os.path.join(root, file))

C:\Users\Asus\Downloads\archive\Retail_Data_Response.csv
C:\Users\Asus\Downloads\archive\Retail_Data_Transactions.csv
C:\Users\Asus\Downloads\archive\.ipynb_checkpoints\Retail_Data_Response-checkpoint.csv
C:\Users\Asus\Downloads\archive\.ipynb_checkpoints\Retail_Data_Transactions-checkpoint.csv
